# 2 — Data Cleaning & Feature Engineering

This notebook prepares the three raw files for analysis. We do only what's necessary:
- Standardise column names
- Convert dollar amounts stored as text into real numbers
- Align the hospital ID across tables (same number, different column names)
- Add MDC (body-system category) names
- Create the two core metrics: **Billing Gap** and **Charge Ratio**

Outputs go to `data/cleaned/` — three CSVs that get loaded into the database in Notebook 3.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA_DIR = Path('..') / 'data'
OUT_DIR  = DATA_DIR / 'cleaned'
OUT_DIR.mkdir(exist_ok=True)

In [2]:
inpatient_raw = pd.read_csv(DATA_DIR / 'Inpatient Payments.CSV')
provider_raw  = pd.read_csv(DATA_DIR / 'Provider Info.csv')
drg_raw       = pd.read_excel(DATA_DIR / 'DRG Details.xls')

print('Shapes — inpatient:', inpatient_raw.shape,
      '| provider:', provider_raw.shape,
      '| drg:', drg_raw.shape)

Shapes — inpatient: (196086, 15) | provider: (5426, 38) | drg: (543, 8)


## Step 1 — Standardise column names

All column names become lowercase with underscores (`snake_case`) so SQL queries are clean and readable.

In [3]:
def snake_case(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return re.sub(r'_+', '_', s).strip('_')

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [snake_case(c) for c in out.columns]
    return out

inpatient = clean_cols(inpatient_raw)
provider  = clean_cols(provider_raw)
drg       = clean_cols(drg_raw)

print('Inpatient cols:', inpatient.columns.tolist())
print('DRG cols:',       drg.columns.tolist())

Inpatient cols: ['rndrng_prvdr_ccn', 'rndrng_prvdr_org_name', 'rndrng_prvdr_city', 'rndrng_prvdr_st', 'rndrng_prvdr_state_fips', 'rndrng_prvdr_zip5', 'rndrng_prvdr_state_abrvtn', 'rndrng_prvdr_ruca', 'rndrng_prvdr_ruca_desc', 'drg_cd', 'drg_desc', 'tot_dschrgs', 'avg_submtd_cvrd_chrg', 'avg_tot_pymt_amt', 'avg_mdcr_pymt_amt']
DRG cols: ['drgv22', 'mdc', 'type', 'unnamed_3', 'drg_title', 'relative_weights', 'geometric_mean_los', 'arithmetic_mean_los']


## Step 2 — Convert dollar fields to numbers

CMS extracts can store money as formatted strings like `$45,230.00`. We strip the `$` and commas so we can do maths on them.

In [4]:
def to_num(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    x = (s.astype(str)
           .str.replace('$', '', regex=False)
           .str.replace(',', '', regex=False)
           .str.strip()
           .replace({'': np.nan, 'nan': np.nan, 'None': np.nan}))
    return pd.to_numeric(x, errors='coerce')

for col in ['avg_submtd_cvrd_chrg', 'avg_tot_pymt_amt', 'avg_mdcr_pymt_amt']:
    inpatient[col] = to_num(inpatient[col])
inpatient['tot_dschrgs'] = pd.to_numeric(inpatient['tot_dschrgs'], errors='coerce')

drg['relative_weights']    = pd.to_numeric(drg['relative_weights'],    errors='coerce')
drg['geometric_mean_los']  = pd.to_numeric(drg['geometric_mean_los'],  errors='coerce')

print('Dtypes confirmed numeric:')
print(inpatient[['avg_submtd_cvrd_chrg', 'avg_mdcr_pymt_amt', 'tot_dschrgs']].dtypes)

Dtypes confirmed numeric:
avg_submtd_cvrd_chrg    float64
avg_mdcr_pymt_amt       float64
tot_dschrgs               int64
dtype: object


## Step 3 — Align the hospital join key

**Problem**: T1 stores the hospital ID as `Rndrng_Prvdr_CCN`, T3 calls it `Facility ID`.
They hold the same 6-digit number but one side might have a leading zero (e.g. `050441` vs `50441`).

**Fix**: Strip leading zeros from both sides so the join always works.

In [5]:
inpatient['facility_id'] = inpatient['rndrng_prvdr_ccn'].astype(str).str.strip().str.lstrip('0')
provider['facility_id']  = provider['facility_id'].astype(str).str.strip().str.lstrip('0')

inpatient['drg_cd'] = pd.to_numeric(inpatient['drg_cd'], errors='coerce').astype('Int64')
drg['drgv22']       = pd.to_numeric(drg['drgv22'],       errors='coerce').astype('Int64')

# Quick join coverage check
matched = inpatient['facility_id'].isin(provider['facility_id']).mean()
print(f'Hospital ID match rate: {matched:.1%}')

Hospital ID match rate: 96.6%


## Step 4 — Add MDC names to DRG details

MDC (Major Diagnostic Category) groups all ~750 procedures into 25 body-system buckets.
e.g. MDC 05 = Heart & Circulatory, MDC 08 = Bones & Joints.

We map the numeric MDC code to a readable name so charts are self-explanatory.

In [6]:
MDC_NAMES = {
    '01': 'Nervous System',
    '02': 'Eye',
    '03': 'Ear, Nose & Throat',
    '04': 'Respiratory System',
    '05': 'Circulatory System (Heart)',
    '06': 'Digestive System',
    '07': 'Liver & Pancreas',
    '08': 'Musculoskeletal (Bones & Joints)',
    '09': 'Skin & Tissue',
    '10': 'Endocrine & Metabolic',
    '11': 'Kidney & Urinary Tract',
    '12': 'Male Reproductive',
    '13': 'Female Reproductive',
    '14': 'Pregnancy & Childbirth',
    '15': 'Newborns',
    '16': 'Blood & Immune Disorders',
    '17': 'Cancer & Tumors',
    '18': 'Infectious Diseases',
    '19': 'Mental Health',
    '20': 'Alcohol & Drug Use',
    '21': 'Injuries & Poisonings',
    '22': 'Burns',
    '23': 'Health Status & Other',
    '24': 'Multiple Trauma',
    '25': 'HIV Infections',
    'PRE': 'Pre-MDC (Transplants & Critical Care)',
}

drg['mdc']      = drg['mdc'].astype(str).str.strip()
drg['mdc_name'] = drg['mdc'].map(MDC_NAMES).fillna('Unknown')
drg['drg_type'] = drg['type'].astype(str).str.strip()

# Drop junk columns and keep only what analysis needs
drg_clean = drg[['drgv22', 'drg_title', 'mdc', 'mdc_name', 'drg_type',
                  'relative_weights', 'geometric_mean_los']].copy()

# Remove zero-weight (invalid/retired) DRGs
drg_clean = drg_clean[drg_clean['relative_weights'] > 0].reset_index(drop=True)
print('Valid DRGs after removing retired entries:', len(drg_clean))
print(drg_clean.head(3).to_string())

Valid DRGs after removing retired entries: 518
   drgv22                  drg_title mdc        mdc_name drg_type  relative_weights  geometric_mean_los
0       1    CRANIOTOMY AGE >17 W CC  01  Nervous System     SURG            3.3344                 7.5
1       2  CRANIOTOMY AGE >17 W/O CC  01  Nervous System     SURG            1.9467                 3.6
2       3        CRANIOTOMY AGE 0-17  01  Nervous System     SURG            1.9767                12.7


## Step 5 — Create the two core analytical metrics

**Billing Gap** (`$`) = what the hospital billed − what Medicare paid.  
**Charge Ratio** (multiplier) = what the hospital billed ÷ what Medicare paid.

- A billing gap of \$40,000 means the hospital billed \$40k more than Medicare paid.
- A charge ratio of 4.5 means the hospital charged 4.5× what Medicare reimbursed.

Both metrics are computed at the row level (hospital × procedure). They get aggregated in the analysis notebooks.

In [7]:
inpatient['billing_gap']  = inpatient['avg_submtd_cvrd_chrg'] - inpatient['avg_mdcr_pymt_amt']
inpatient['charge_ratio'] = inpatient['avg_submtd_cvrd_chrg'] / inpatient['avg_mdcr_pymt_amt']
inpatient.loc[inpatient['avg_mdcr_pymt_amt'] <= 0, 'charge_ratio'] = np.nan

print('Billing gap stats ($ per hospital-DRG row):')
print(inpatient['billing_gap'].describe().round(0))
print('\nCharge ratio stats (× multiplier):')
print(inpatient['charge_ratio'].describe().round(2))

Billing gap stats ($ per hospital-DRG row):
count     196086.0
mean       50082.0
std        63692.0
min       -23225.0
25%        18018.0
50%        32194.0
75%        58902.0
max      2865308.0
Name: billing_gap, dtype: float64

Charge ratio stats (× multiplier):
count    196086.00
mean          5.59
std           3.01
min           0.20
25%           3.60
50%           4.90
75%           6.86
max          70.92
Name: charge_ratio, dtype: float64


**What this tells us right away**: The median charge ratio is around 4–5×, meaning a typical hospital bills 4–5 times what Medicare pays. Some extreme rows go up to 70×.

## Step 6 — Missing value check

In [8]:
for name, df in [('inpatient', inpatient), ('provider', provider), ('drg', drg_clean)]:
    missing = df.isna().mean().sort_values(ascending=False)
    missing = missing[missing > 0]
    if len(missing):
        print(f'\n{name} — missing value rates:')
        print(missing.round(3).to_string())
    else:
        print(f'\n{name} — no missing values in key columns')


inpatient — missing value rates:
rndrng_prvdr_ruca         0.0
rndrng_prvdr_ruca_desc    0.0

provider — missing value rates:
te_group_footnote                                   0.827
readm_group_footnote                                0.786
mort_group_footnote                                 0.671
safety_group_footnote                               0.617
meets_criteria_for_birthing_friendly_designation    0.583
pt_exp_group_footnote                               0.581
hospital_overall_rating_footnote                    0.526

drg — missing value rates:
mdc         0.006
drg_type    0.002


## Step 7 — Select only the columns used in analysis and save

In [9]:
inpatient_cols = [
    'rndrng_prvdr_ccn', 'rndrng_prvdr_org_name',
    'rndrng_prvdr_city', 'rndrng_prvdr_st', 'rndrng_prvdr_state_abrvtn',
    'drg_cd', 'drg_desc', 'tot_dschrgs',
    'avg_submtd_cvrd_chrg', 'avg_tot_pymt_amt', 'avg_mdcr_pymt_amt',
    'facility_id', 'billing_gap', 'charge_ratio'
]
provider_cols = [
    'facility_id', 'facility_name', 'address',
    'city_town', 'state', 'zip_code',
    'hospital_type', 'hospital_ownership', 'emergency_services'
]

inpatient[inpatient_cols].to_csv(OUT_DIR / 'inpatient_payments_cleaned.csv', index=False)
provider[provider_cols].to_csv(OUT_DIR / 'provider_info_cleaned.csv', index=False)
drg_clean.to_csv(OUT_DIR / 'drg_details_cleaned.csv', index=False)

# Also save a master joined file for Python-only notebooks
master = (inpatient[inpatient_cols]
    .merge(provider[provider_cols], on='facility_id', how='left')
    .merge(drg_clean, left_on='drg_cd', right_on='drgv22', how='left'))
master.to_csv(OUT_DIR / 'master_inpatient_payments_cleaned.csv', index=False)

print('Saved cleaned CSVs to:', OUT_DIR.resolve())
print(f'  inpatient: {len(inpatient):,} rows')
print(f'  provider:  {len(provider):,} rows')
print(f'  drg:       {len(drg_clean):,} rows')
print(f'  master:    {len(master):,} rows')

Saved cleaned CSVs to: /Users/apekshaa/Desktop/Healthcare_Analysis-main/data/cleaned
  inpatient: 196,086 rows
  provider:  5,426 rows
  drg:       518 rows
  master:    196,086 rows
